In [2]:
"""
Milestone 2: Tracked evaluation runner (sweep + locked threshold + test).

Pipeline flow:
  prepare_pairs.py (M1) -> saves pair artifact
  -> run_eval.py loads pair artifact
  -> validates input
  -> computes or loads scores
  -> runs threshold sweep or fixed-threshold evaluation
  -> saves metrics / confusion matrix / plot / predictions
  -> appends a run row to run_summary.csv
  -> (optional) run_error_analysis.py loads saved predictions

This script is intentionally thin. It orchestrates the evaluation by calling
reusable functions from `src/` so the logic is testable.
"""

from __future__ import annotations

import argparse
import sys
from pathlib import Path
from typing import Any, Optional

# ---------------------------------------------------------------------------
# Project root setup (notebook-safe: __file__ is not defined in Jupyter)
# ---------------------------------------------------------------------------
import os
if "__file__" in globals():
    PROJECT_ROOT = Path(__file__).resolve().parents[1]
else:
    PROJECT_ROOT = Path(os.getcwd()).resolve()
    # If running from scripts/, go up one level to project root
    if PROJECT_ROOT.name == "scripts":
        PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import Config
from src.tracking import (
    make_run_id,              # ✅ generates unique run id
    get_timestamp,            # ✅ UTC timestamp string
    get_git_commit_hash,      # ✅ current HEAD sha
    create_run_dir,           # ✅ creates outputs/runs/<run_id>/
    save_run_info,            # ❌ TODO: write run_info.json
    append_run_summary,       # ❌ TODO: append row to outputs/run_summary.csv
)

# ---------------------------------------------------------------------------
# Imports: I/O utilities  (all ❌ TODO stubs)
# ---------------------------------------------------------------------------
from src.io_utils import (
    load_pairs_csv,           # ❌ TODO: load pair artifact CSV into memory
    save_csv,                 # ❌ TODO: generic CSV writer
    save_json,                # ❌ TODO: generic JSON writer
    copy_config_snapshot,     # ❌ TODO: snapshot config YAML into run dir
)

# ---------------------------------------------------------------------------
# Imports: validation  (all ❌ TODO stubs)
# ---------------------------------------------------------------------------
from src.validation import (
    validate_pairs_df,        # ❌ TODO: schema / missing-value checks
    validate_image_paths,     # ❌ TODO: confirm image files exist
    validate_threshold_config,# ❌ TODO: validate threshold mode / range
    validate_scores_length,   # ❌ TODO: scores count == pairs count
    check_split_integrity,    # ❌ TODO: no leakage / valid split labels
)

# ---------------------------------------------------------------------------
# Imports: scoring & evaluation  (all ❌ TODO stubs)
# ---------------------------------------------------------------------------
from src.evaluation import (
    load_or_compute_scores,   # ❌ TODO: load saved scores or compute via M1 pipeline
    evaluate_at_threshold,    # ❌ TODO: threshold -> predictions -> metrics dict
    run_threshold_sweep,      # ❌ TODO: loop thresholds, record metrics per threshold
)

# ---------------------------------------------------------------------------
# Imports: thresholding  (all ❌ TODO stubs)
# ---------------------------------------------------------------------------
from src.thresholding import (
    generate_threshold_grid,      # ❌ TODO: build list of thresholds from config
    select_threshold,             # ❌ TODO: pick best threshold from sweep table
    is_higher_score_same_person,  # ❌ TODO: score direction convention
)

# ---------------------------------------------------------------------------
# Imports: plotting  (all ❌ TODO stubs)
# ---------------------------------------------------------------------------
from src.plotting import (
    plot_roc_style_curve,     # ❌ TODO: ROC-style FPR vs TPR plot
)

def parse_args(argv: Optional[list[str]] = None) -> argparse.Namespace:
    """
    Reads --config and optional --note.

    --config : path to YAML/JSON config
    --note   : optional short note recorded in run_info + run_summary
    """
    p = argparse.ArgumentParser(description="Run tracked evaluation for face verification.")
    p.add_argument(
        "--config",
        required=True,
        help="Path to an evaluation config (YAML or JSON).",
    )
    p.add_argument(
        "--note",
        default=None,
        help="Optional short note to attach to this run (stored in run artifacts).",
    )
    return p.parse_args(argv)


# ═══════════════════════════════════════════════════════════════════════════
# Pretty-print helper
# ═══════════════════════════════════════════════════════════════════════════

def print_run_summary(
    *,
    run_id: str,
    run_dir: Path,
    mode: str,
    split: str,
    selected_threshold: Optional[float],
    metrics: dict[str, Any],
) -> None:
    """Print final run id, threshold, metrics, and output path."""
    print("\n=== RUN COMPLETE ===")
    print(f"  run_id:    {run_id}")
    print(f"  mode:      {mode}")
    print(f"  split:     {split}")
    print(f"  run_dir:   {run_dir}")
    if selected_threshold is not None:
        print(f"  threshold: {selected_threshold}")
    print("  metrics:")
    for k in sorted(metrics.keys()):
        print(f"    {k}: {metrics[k]}")
    print("====================\n")


# ═══════════════════════════════════════════════════════════════════════════
# Main pipeline
# ═══════════════════════════════════════════════════════════════════════════

def main(argv: Optional[list[str]] = None) -> int:
    """
    Pipeline stages (in order):
      1. Parse args & load config
      2. Create run metadata (run_id, timestamp, git hash)
      3. Create run directory & snapshot config + run_info.json
      4. Load pair artifact & validate inputs
      5. Load or compute similarity scores & validate alignment
      6. Threshold sweep  -OR-  fixed threshold
      7. Evaluate at selected threshold -> metrics, confusion, predictions
      8. Persist all artifacts to run directory
      9. Append one row to master run_summary.csv
     10. Print summary to terminal
    """

    # ------------------------------------------------------------------
    # Stage 1: Parse args & load config
    # ------------------------------------------------------------------
    args = parse_args(argv)
    config_path = Path(args.config).expanduser()
    note = args.note

    config = Config.from_file(config_path)

    # ------------------------------------------------------------------
    # Stage 2: Run identity & provenance  (✅ implemented)
    # ------------------------------------------------------------------
    run_id = make_run_id()
    timestamp = get_timestamp()
    commit_hash = get_git_commit_hash()

    # ------------------------------------------------------------------
    # Stage 3: Create run directory & snapshot config  (partially ✅)
    # ------------------------------------------------------------------
    run_dir = create_run_dir(config, run_id)       # ✅

    # TODO: implement copy_config_snapshot in src/io_utils.py
    # copy_config_snapshot(config, run_dir / "config_used.yaml")

    run_info = {
        "run_id": run_id,
        "timestamp": timestamp,
        "commit_hash": commit_hash,
        "config_path": str(config_path),
        "note": note,
    }
    # TODO: implement save_run_info in src/tracking.py
    # save_run_info(run_dir / "run_info.json", run_info)

    # ------------------------------------------------------------------
    # Stage 4: Load pair artifact & validate inputs
    # ------------------------------------------------------------------
    # TODO: implement load_pairs_csv in src/io_utils.py
    train_pairs_df, val_pairs_df, test_pairs_df = load_pairs_csv(config)

#     # TODO: implement all validators in src/validation.py
#     # validate_pairs_df(train_pairs_df)
#     # validate_pairs_df(val_pairs_df)
#     # validate_pairs_df(test_pairs_df)
#     # validate_image_paths(train_pairs_df, config)
#     # validate_image_paths(val_pairs_df, config)
#     # validate_image_paths(test_pairs_df, config)
#     # check_split_integrity(train_pairs_df)
#     # check_split_integrity(val_pairs_df)
#     # check_split_integrity(test_pairs_df)

#     # ------------------------------------------------------------------
#     # Stage 5: Filter to eval split & load / compute scores
#     # ------------------------------------------------------------------
#     eval_split = getattr(config, "split", None)
#     if eval_split is None:
#         raise ValueError("Missing config.eval.split (expected 'train'|'val'|'test').")

    
#     # TODO: implement filter_to_split (below in this file)
#     # eval_df = filter_to_split(pairs_df, eval_split)

#     # TODO: implement load_or_compute_scores in src/evaluation.py
#     # scores = load_or_compute_scores(eval_df, config)

#     # TODO: implement validate_scores_length in src/validation.py
#     # validate_scores_length(eval_df, scores)

#     # ------------------------------------------------------------------
#     # Stage 5b: Validate threshold config
#     # ------------------------------------------------------------------
#     threshold_cfg = getattr(config, "threshold", None)
#     # TODO: implement validate_threshold_config in src/validation.py
#     # validate_threshold_config(threshold_cfg)

#     # TODO: implement is_higher_score_same_person in src/thresholding.py
#     # higher_means_same = is_higher_score_same_person(config)

#     # ------------------------------------------------------------------
#     # Stage 6: Threshold sweep  -OR-  fixed threshold
#     # ------------------------------------------------------------------
#     mode = getattr(getattr(config, "eval", None), "mode", None)
#     if mode is None:
#         raise ValueError("Missing config.eval.mode (expected 'sweep' or 'fixed').")

#     selected_threshold: Optional[float] = None
#     # sweep_df = None

#     if mode == "sweep":
#         # TODO: implement generate_threshold_grid in src/thresholding.py
#         # thresholds = generate_threshold_grid(config)

#         # TODO: implement run_threshold_sweep in src/evaluation.py
#         # sweep_df = run_threshold_sweep(eval_df, thresholds, higher_means_same=higher_means_same)

#         # TODO: implement save_sweep_table (below in this file)
#         # save_sweep_table(sweep_df, run_dir / "threshold_sweep.csv")

#         # TODO: implement select_threshold in src/thresholding.py
#         # rule = getattr(threshold_cfg, "selection_rule", "max_balanced_accuracy")
#         # selected_threshold = select_threshold(sweep_df, rule)

#         # TODO: implement save_json in src/io_utils.py
#         # save_json(
#         #     {"selected_threshold": selected_threshold, "selection_rule": rule},
#         #     run_dir / "selected_threshold.json",
#         # )

#         # TODO: implement plot_roc_style_curve in src/plotting.py
#         # plot_roc_style_curve(sweep_df, run_dir / "roc_curve.png")
#         pass

#     elif mode == "fixed":
#         selected_threshold = getattr(threshold_cfg, "fixed_value", None)
#         if selected_threshold is None:
#             raise ValueError("Fixed mode requires config.threshold.fixed_value.")
#     else:
#         raise ValueError(f"Unknown eval mode: {mode!r} (expected 'sweep' or 'fixed').")

#     # ------------------------------------------------------------------
#     # Stage 7: Evaluate at the chosen threshold
#     #
#     # evaluate_at_threshold should internally use src/metrics.py:
#     #   - apply_threshold()           -> y_pred
#     #   - compute_confusion_counts()  -> (TP, FP, TN, FN)
#     #   - summarize_metrics()         -> full metrics dict
#     #   - attach_predictions()        -> predictions table
#     #
#     # Expected return:
#     #   {
#     #     "metrics":     { accuracy, balanced_accuracy, precision, recall, f1_score, confusion_matrix },
#     #     "confusion":   { TP, FP, TN, FN },
#     #     "predictions": <table: y_true, score, y_pred, threshold, left_path, right_path>
#     #   }
#     # ------------------------------------------------------------------
#     # TODO: implement evaluate_at_threshold in src/evaluation.py
#     # eval_out = evaluate_at_threshold(
#     #     eval_df,
#     #     threshold=selected_threshold,
#     #     higher_means_same=higher_means_same,
#     # )
#     # metrics        = eval_out["metrics"]
#     # confusion      = eval_out["confusion"]
#     # predictions_df = eval_out["predictions"]

#     # ------------------------------------------------------------------
#     # Stage 8: Persist artifacts to run directory
#     # ------------------------------------------------------------------
#     # TODO: implement save_json in src/io_utils.py
#     # save_json(metrics,   run_dir / "metrics.json")
#     # save_json(confusion, run_dir / "confusion_matrix.json")

#     # TODO: implement save_predictions (below in this file)
#     # save_predictions(predictions_df, run_dir / "predictions.csv")

#     # ------------------------------------------------------------------
#     # Stage 9: Append row to master run_summary.csv
#     # ------------------------------------------------------------------
#     # TODO: implement append_run_summary in src/tracking.py
#     # append_run_summary(config, {
#     #     "run_id": run_id,
#     #     "timestamp": timestamp,
#     #     "commit_hash": commit_hash,
#     #     "config_path": str(config_path),
#     #     "mode": mode,
#     #     "split": eval_split,
#     #     "selected_threshold": selected_threshold,
#     #     "note": note,
#     #     "metrics": metrics,
#     # })

#     # ------------------------------------------------------------------
#     # Stage 10: Print summary to terminal
#     # ------------------------------------------------------------------
#     # print_run_summary(
#     #     run_id=run_id,
#     #     run_dir=run_dir,
#     #     mode=mode,
#     #     split=eval_split,
#     #     selected_threshold=selected_threshold,
#     #     metrics=metrics,
#     # )
#     return 0


# # ═══════════════════════════════════════════════════════════════════════════
# # Local helpers (TODO: implement each)
# # ═══════════════════════════════════════════════════════════════════════════

# # def filter_to_split(pairs_df: Any, split: str) -> Any:
# #     """Return only rows belonging to the requested `split` (train/val/test)."""
# #     raise NotImplementedError("TODO: implement filter_to_split()")


# # def save_sweep_table(sweep_df: Any, out_path: Path) -> None:
# #     """Persist threshold sweep results to CSV."""
# #     raise NotImplementedError("TODO: implement save_sweep_table()")


# # def save_predictions(predictions_df: Any, out_path: Path) -> None:
# #     """Persist predictions (y_true, score, y_pred, threshold, paths) to CSV."""
# #     raise NotImplementedError("TODO: implement save_predictions()")


if __name__ == "__main__":
    raise SystemExit(main())


usage: ipykernel_launcher.py [-h] --config CONFIG [--note NOTE]
ipykernel_launcher.py: error: the following arguments are required: --config


SystemExit: 2

/Users/nandanprince/venvs/emb/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [5]:
from src.io_utils import (
    load_pairs_csv,           # ❌ TODO: load pair artifact CSV into memory
    save_csv,                 # ❌ TODO: generic CSV writer
    save_json,                # ❌ TODO: generic JSON writer
    copy_config_snapshot,     # ❌ TODO: snapshot config YAML into run dir
)
from src.config import Config
config_path = PROJECT_ROOT / "configs" / "default.yaml"
config = Config.from_file(config_path)
train_pairs_df, val_pairs_df, test_pairs_df = load_pairs_csv(config)

FileNotFoundError: [Errno 2] No such file or directory: 'outputs/pairs/train_pairs.csv'